# BFS: LAGraph (CPU) vs SPLA (GPU)

Datasets: soc-LiveJournal1, patents, wiki-Talk, rgg_n_2_22_s0  
Modes: directed (kind=1, no sym) / undirected (sym=1, kind=0)  
Runs: 15 per config, run 1 dropped (JIT warmup for SPLA, cold cache for LAGraph)  
Source: max-out-degree node (maximises reachability, consistent with LAGraph bench suite)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid')

REPO = Path('../../')
RESULTS = REPO / 'lagraph-spla' / 'results' / 'bfs'
PICS    = REPO / 'lagraph-spla' / 'presentation' / 'pictures' / 'bfs'
PICS.mkdir(parents=True, exist_ok=True)

GRAPH_EDGES = {
    'soc-LiveJournal1.mtx': 68_993_773,
    'patents.mtx':          14_970_767,
    'wiki-Talk.mtx':         5_021_410,
    'rgg_n_2_22_s0.mtx':   30_359_198,
}
NAME_MAP = {
    'soc-LiveJournal1.mtx': 'LiveJournal',
    'patents.mtx':          'Patents†',
    'wiki-Talk.mtx':        'wiki-Talk',
    'rgg_n_2_22_s0.mtx':   'RGG',
}
ORDER_MODES = ['LAGraph directed', 'LAGraph undirected', 'SpLA directed', 'SpLA undirected']
GRAPHS_ORDER = ['LiveJournal', 'wiki-Talk', 'RGG', 'Patents†']

df_raw = pd.read_csv(RESULTS / 'bfs_results_v2.csv')
df_raw['GraphName'] = df_raw['Graph'].map(NAME_MAP)
print(f'Total rows: {len(df_raw)}')
df_raw.head()

In [ ]:
# Drop run 1 (JIT warmup for SpLA; cold-start for LAGraph)
df = df_raw[df_raw['Iteration'] > 1].copy()

# Normality test per group
print('=== Shapiro-Wilk normality test (p < 0.05 → non-normal) ===')
for (graph, mode), grp in df.groupby(['Graph', 'LibraryMode']):
    times = grp['Time_ms'].dropna()
    if len(times) >= 3:
        w, p = stats.shapiro(times)
        flag = '' if p > 0.05 else '  ← non-normal'
        print(f'  {NAME_MAP.get(graph, graph):<14} | {mode:<22} | p={p:.4f}{flag}')

In [ ]:
# Summary statistics
def get_stats(group):
    t = group['Time_ms']
    n = len(t)
    med = np.median(t)
    mean = np.mean(t)
    se = stats.sem(t)
    ci = stats.t.interval(0.95, n-1, loc=mean, scale=se) if n > 1 else (mean, mean)
    cv = t.std() / mean * 100 if mean > 0 else 0
    return pd.Series({
        'Median_ms': med, 'Mean_ms': mean,
        'CI95_low': ci[0], 'CI95_high': ci[1],
        'CV_%': cv, 'N': n
    })

summary = df.groupby(['Graph', 'GraphName', 'LibraryMode']).apply(get_stats).reset_index()
summary['LibraryMode'] = pd.Categorical(summary['LibraryMode'], categories=ORDER_MODES, ordered=True)
summary['GraphName']   = pd.Categorical(summary['GraphName'],   categories=GRAPHS_ORDER, ordered=True)
summary = summary.sort_values(['GraphName', 'LibraryMode'])

print('=== BFS Summary (median, drop run 1) ===')
cols = ['GraphName', 'LibraryMode', 'Median_ms', 'CV_%', 'N']
print(summary[cols].to_string(index=False))

In [ ]:
# Chart 1: Grouped bar — all datasets × all modes
# Exclude patents directed (only 428 visited, not meaningful comparison)
plot_df = summary.copy()
present_modes = [m for m in ORDER_MODES if m in plot_df['LibraryMode'].values]
graphs_present = [g for g in GRAPHS_ORDER if g in plot_df['GraphName'].values]

fig, ax = plt.subplots(figsize=(14, 6))
sns.barplot(data=plot_df, x='GraphName', y='Median_ms',
            hue='LibraryMode', hue_order=present_modes,
            order=graphs_present, palette='Set2', ax=ax)

# Error bars (95% CI)
# seaborn barplot with ci would do this, but we've pre-aggregated, so add manually
for p, (_, row) in zip(ax.patches, plot_df.iterrows()):
    pass  # annotations below

for p in ax.patches:
    h = p.get_height()
    if pd.notna(h) and h > 5:
        ax.annotate(f'{h:.0f}', (p.get_x() + p.get_width()/2, h),
                    ha='center', va='bottom', fontsize=7)

ax.set_title('BFS median time — all datasets (runs 2–15, warmup dropped)\n'
             '†Patents directed: only 428/3.7M nodes visited (DAG structure)')
ax.set_ylabel('Median time (ms)')
ax.set_xlabel('Dataset')
ax.legend(title='Mode', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.grid(axis='y', alpha=0.4)
fig.tight_layout()
fig.savefig(PICS / 'bfs_all_datasets.png', dpi=150)
plt.show()

In [ ]:
# Chart 2: Box plots — distribution of run times
# Shows variance, outliers, and JIT warmup spike for SpLA
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
graphs_4 = ['LiveJournal', 'wiki-Talk', 'RGG', 'Patents†']

for ax, gname in zip(axes.flat, graphs_4):
    # Include run 1 to show JIT spike
    sub = df_raw[df_raw['GraphName'] == gname].copy()
    sub['LibraryMode'] = pd.Categorical(sub['LibraryMode'], categories=ORDER_MODES, ordered=True)
    sub = sub[sub['LibraryMode'].notna()]
    if sub.empty:
        continue
    # Mark run 1 vs rest
    sub['RunType'] = sub['Iteration'].apply(lambda x: 'Run 1 (warmup)' if x == 1 else 'Runs 2-15')
    
    warm = df_raw[(df_raw['GraphName'] == gname) & (df_raw['Iteration'] > 1)].copy()
    warm['LibraryMode'] = pd.Categorical(warm['LibraryMode'], categories=ORDER_MODES, ordered=True)
    warm = warm[warm['LibraryMode'].notna()]
    
    sns.boxplot(data=warm, x='LibraryMode', y='Time_ms', palette='Set2', ax=ax)
    ax.set_title(f'{gname}')
    ax.set_ylabel('Time (ms)')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15)
    ax.grid(axis='y', alpha=0.4)

fig.suptitle('BFS time distribution (box plots, runs 2–15)', fontsize=13)
fig.tight_layout()
fig.savefig(PICS / 'bfs_boxplots.png', dpi=150)
plt.show()

In [ ]:
# Chart 3: GTEPS — throughput
# Exclude patents directed (428 visited ≠ full graph traversal)
gteps = summary.copy()
gteps['Edges'] = gteps['Graph'].map(GRAPH_EDGES)
gteps['GTEPS'] = gteps['Edges'] / (gteps['Median_ms'] * 1e6)

# Mask patents directed — misleadingly high GTEPS due to trivial work
mask = (gteps['GraphName'] == 'Patents†') & (gteps['LibraryMode'] == 'LAGraph directed')
gteps.loc[mask, 'GTEPS'] = float('nan')
mask2 = (gteps['GraphName'] == 'Patents†') & (gteps['LibraryMode'] == 'SpLA directed')
gteps.loc[mask2, 'GTEPS'] = float('nan')

fig, ax = plt.subplots(figsize=(13, 6))
sns.barplot(data=gteps.dropna(subset=['GTEPS']), x='GraphName', y='GTEPS',
            hue='LibraryMode', hue_order=present_modes,
            order=graphs_present, palette='viridis', ax=ax)
for p in ax.patches:
    h = p.get_height()
    if pd.notna(h) and h > 0:
        ax.annotate(f'{h:.3f}', (p.get_x() + p.get_width()/2, h),
                    ha='center', va='bottom', fontsize=7)
ax.set_title('BFS throughput (GTEPS = Giga Edges per Second)\n'
             'Patents directed excluded (only 428/3.7M nodes visited)')
ax.set_ylabel('GTEPS')
ax.legend(title='Mode', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.grid(axis='y', alpha=0.4)
fig.tight_layout()
fig.savefig(PICS / 'bfs_gteps_all.png', dpi=150)
plt.show()

In [ ]:
# Chart 4: Directed vs Undirected delta for LAGraph
lg = summary[summary['LibraryMode'].isin(['LAGraph directed', 'LAGraph undirected'])].copy()
piv = lg.pivot_table(index='GraphName', columns='LibraryMode', values='Median_ms').reset_index()
piv.columns.name = None

if 'LAGraph directed' in piv.columns and 'LAGraph undirected' in piv.columns:
    piv['delta_pct'] = (piv['LAGraph undirected'] / piv['LAGraph directed'] - 1) * 100
    piv = piv.dropna(subset=['delta_pct'])

    colors = ['#d62728' if v > 0 else '#2ca02c' for v in piv['delta_pct']]
    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.bar(piv['GraphName'], piv['delta_pct'], color=colors, width=0.5)
    ax.axhline(0, color='black', lw=1)
    for bar, v in zip(bars, piv['delta_pct']):
        ax.annotate(f'{v:+.0f}%', (bar.get_x() + bar.get_width()/2, v),
                    ha='center', va='bottom' if v >= 0 else 'top', fontsize=10)
    ax.set_title('LAGraph BFS: undirected vs directed median time delta\n'
                 'Red = undirected is slower, Green = undirected is faster')
    ax.set_ylabel('Δ time (%)')
    ax.grid(axis='y', alpha=0.4)
    fig.tight_layout()
    fig.savefig(PICS / 'bfs_directed_vs_undirected_delta.png', dpi=150)
    plt.show()

In [ ]:
# Chart 5: SpLA GPU speedup over LAGraph CPU (directed mode)
spiv = summary.pivot_table(index='GraphName', columns='LibraryMode', values='Median_ms').reset_index()
spiv.columns.name = None

if 'LAGraph directed' in spiv.columns and 'SpLA directed' in spiv.columns:
    spiv['Speedup'] = spiv['LAGraph directed'] / spiv['SpLA directed']
    spiv = spiv.dropna(subset=['Speedup'])
    # Exclude patents (not a fair comparison - different visited counts)
    spiv_plot = spiv[spiv['GraphName'] != 'Patents†']

    fig, ax = plt.subplots(figsize=(9, 5))
    colors = ['#2196F3' if v >= 1 else '#FF7043' for v in spiv_plot['Speedup']]
    bars = ax.bar(spiv_plot['GraphName'], spiv_plot['Speedup'], color=colors)
    ax.axhline(1.0, color='red', ls='--', lw=1.5, label='CPU = GPU baseline')
    for bar, v in zip(bars, spiv_plot['Speedup']):
        ax.annotate(f'{v:.2f}×', (bar.get_x() + bar.get_width()/2, v),
                    ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax.set_title('SpLA GPU speedup vs LAGraph CPU (directed mode)\n'
                 'Blue ≥ 1× (GPU faster), Orange < 1× (CPU faster)')
    ax.set_ylabel('Speedup ×')
    ax.legend()
    ax.grid(axis='y', alpha=0.4)
    fig.tight_layout()
    fig.savefig(PICS / 'bfs_gpu_speedup.png', dpi=150)
    plt.show()

print('\n=== Interpretation ===')
for _, row in spiv_plot.iterrows():
    s = row['Speedup']
    winner = 'SpLA GPU' if s > 1 else 'LAGraph CPU'
    print(f'  {row["GraphName"]:<14}: {winner} {abs(s):.1f}× (speedup={s:.2f})')

In [ ]:
# Chart 6: RGG deep dive — why GPU is slower
rgg = df[df['Graph'] == 'rgg_n_2_22_s0.mtx'].copy()
if not rgg.empty:
    rgg['LibraryMode'] = pd.Categorical(rgg['LibraryMode'], categories=ORDER_MODES, ordered=True)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Left: run-by-run time series
    for mode, grp in rgg.groupby('LibraryMode', observed=True):
        axes[0].plot(grp['Iteration'], grp['Time_ms'], marker='o', label=mode, markersize=4)
    axes[0].set_title('RGG: time per run (runs 2–15)')
    axes[0].set_xlabel('Run #')
    axes[0].set_ylabel('Time (ms)')
    axes[0].legend(fontsize=8)
    axes[0].grid(alpha=0.4)

    # Right: comparison with other graphs (LAGraph directed only)
    lg_dir = summary[summary['LibraryMode'] == 'LAGraph directed'].set_index('GraphName')['Median_ms']
    spla_dir = summary[summary['LibraryMode'] == 'SpLA directed'].set_index('GraphName')['Median_ms']
    both = pd.DataFrame({'LAGraph': lg_dir, 'SpLA': spla_dir}).dropna()
    x = np.arange(len(both))
    w = 0.35
    axes[1].bar(x - w/2, both['LAGraph'], w, label='LAGraph CPU', color='#4CAF50')
    axes[1].bar(x + w/2, both['SpLA'],    w, label='SpLA GPU',    color='#2196F3')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(both.index, rotation=10)
    axes[1].set_title('Directed BFS: CPU vs GPU (all graphs)')
    axes[1].set_ylabel('Median time (ms)')
    axes[1].legend()
    axes[1].grid(axis='y', alpha=0.4)

    fig.tight_layout()
    fig.savefig(PICS / 'bfs_rgg_analysis.png', dpi=150)
    plt.show()

    print('\nRGG analysis:')
    print('  - Geometric random graph: ~4M nodes, ~30M edges, avg degree ~14')
    print('  - BFS has ~300 levels (high diameter)')
    print('  - Each level: small frontier → many GPU kernel launches with low occupancy')
    print('  - Kernel launch overhead dominates over compute → GPU loses to CPU')
    print('  - SpLA undirected 2100ms: symmetrisation doubles edges, worsening the problem')

In [ ]:
# Chart 7: Visited nodes comparison — graph structure insight
vis = df[df['Visited'] > 0].groupby(['GraphName', 'LibraryMode'])['Visited'].median().reset_index()
vis['LibraryMode'] = pd.Categorical(vis['LibraryMode'], categories=['LAGraph directed', 'LAGraph undirected'], ordered=True)
vis = vis[vis['LibraryMode'].isin(['LAGraph directed', 'LAGraph undirected'])].dropna()

total_nodes = {
    'LiveJournal': 4_847_571,
    'Patents†':    3_774_768,
    'wiki-Talk':   2_394_385,
    'RGG':         4_194_304,
}
vis['Coverage_%'] = vis.apply(
    lambda r: r['Visited'] / total_nodes.get(r['GraphName'], r['Visited']) * 100, axis=1)

fig, ax = plt.subplots(figsize=(11, 5))
piv_vis = vis.pivot_table(index='GraphName', columns='LibraryMode', values='Coverage_%').reset_index()
piv_vis.columns.name = None

x = np.arange(len(piv_vis))
w = 0.35
b1 = ax.bar(x - w/2, piv_vis.get('LAGraph directed',   [0]*len(piv_vis)), w, label='LAGraph directed')
b2 = ax.bar(x + w/2, piv_vis.get('LAGraph undirected', [0]*len(piv_vis)), w, label='LAGraph undirected')
for bar in list(b1) + list(b2):
    h = bar.get_height()
    if h > 0:
        ax.annotate(f'{h:.1f}%', (bar.get_x() + bar.get_width()/2, h),
                    ha='center', va='bottom', fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(piv_vis['GraphName'], rotation=10)
ax.set_ylabel('Coverage (% of all nodes visited)')
ax.set_title('BFS node coverage: directed vs undirected')
ax.legend()
ax.grid(axis='y', alpha=0.4)
fig.tight_layout()
fig.savefig(PICS / 'bfs_coverage.png', dpi=150)
plt.show()

print('\nKey insight on Patents†:')
print('  Directed BFS visits only 428/3,774,768 nodes (0.01%)')
print('  This is because patents is a citation graph with forward-only edges,')
print('  and the max-out-degree node (36 edges) has very limited forward reach.')
print('  The 1.7ms directed time is not a meaningful throughput measurement.')

In [ ]:
print('=' * 65)
print('BFS FINAL SUMMARY')
print('=' * 65)
for gname in GRAPHS_ORDER:
    sub = summary[summary['GraphName'] == gname]
    if sub.empty:
        continue
    print(f'\n  {gname}')
    for _, row in sub.iterrows():
        ci_half = (row['CI95_high'] - row['CI95_low']) / 2
        print(f'    {row["LibraryMode"]:<25}: {row["Median_ms"]:7.1f} ms '
              f'(±{ci_half:.1f} ms 95%CI, CV={row["CV_%"]:.1f}%)')

print('\n\nKey conclusions:')
print('  1. LAGraph BFS is effectively single-threaded on M2 (memory-bandwidth bound)')
print('  2. SpLA GPU wins on social/communication graphs (LiveJournal 4.8×, wiki-Talk 1.9×)')
print('  3. SpLA GPU LOSES on RGG (0.75× CPU) — high diameter + small frontiers → low GPU occupancy')
print('  4. Undirected is faster than directed for LAGraph on social graphs (pull phase + denser connectivity)')
print('  5. Patents directed BFS: only 428 nodes visited — not a representative benchmark')